# Notebook 03 — PDB Format and Structure Parsing

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 03 of 08  
**Time estimate:** 75 minutes

> The Protein Data Bank stores >215,000 experimental structures.
> Every computational structural biology task starts with parsing a PDB file.
> This notebook implements a PDB parser from scratch — the most practical
> coding skill in this module.

---
## Step 1 — Motivation

PDB parsing is a common coding interview question for bioinformatics roles.
Understanding the format also builds intuition for how structure data is
organized — chain hierarchy, residue numbering, alternate conformations,
heteroatoms — which affects every downstream analysis.

---
## Step 2 — Intuition

A PDB file is a fixed-column text file (legacy format from 1970s). Each line
is called a **record** identified by a 6-character code in columns 1–6.

Key records:
- **ATOM:** standard amino acid atoms — the main data
- **HETATM:** non-standard residues (ligands, water, cofactors)
- **SEQRES:** sequence listed as 3-letter residue codes
- **REMARK:** metadata, resolution, R-factors
- **END:** end of file

The modern replacement is **mmCIF** (macromolecular Crystallographic Information
File) — a key-value format; Biopython's `MMCIFParser` handles it. The legacy PDB
format has a 9,999-residue limit; mmCIF does not.

---
## Step 3 — Biological Background

**ATOM record column layout (fixed-width):**

```
Columns  Content
1–6      Record type ("ATOM  " or "HETATM")
7–11     Atom serial number
13–16    Atom name (e.g. " CA " for Cα)
17       Alternate location indicator
18–20    Residue name (3-letter, e.g. "ALA")
22       Chain ID
23–26    Residue sequence number
27       Code for insertion of residues
31–38    X coordinate (Å, 8.3f format)
39–46    Y coordinate (Å, 8.3f format)
47–54    Z coordinate (Å, 8.3f format)
55–60    Occupancy (1.00 = fully occupied)
61–66    B-factor (temperature factor)
77–78    Element symbol
79–80    Charge on atom
```

**Structure hierarchy:**  
PDB structure → Models (for NMR ensembles) → Chains → Residues → Atoms

**Alternate conformations:** Some residues have two conformations (altloc A/B).
Typically take altloc A or the highest-occupancy conformation.

**HETATM vs ATOM:** Water molecules (HOH) and small molecule ligands are HETATM.
They are usually excluded from backbone geometry calculations.

---
## Step 4 — Mathematical Explanation

No new mathematics in this notebook — the PDB format is a data structure, not an
algorithm. The key computational concept is **coordinate geometry** in 3D.

**Distance between two atoms:**
$$d = \|\mathbf{r}_i - \mathbf{r}_j\| = \sqrt{(x_i-x_j)^2 + (y_i-y_j)^2 + (z_i-z_j)^2}$$

**Radius of gyration** (a measure of compactness):
$$R_g = \sqrt{\frac{1}{N} \sum_{i=1}^N \|\mathbf{r}_i - \bar{\mathbf{r}}\|^2}$$

where $\bar{\mathbf{r}}$ is the centroid of all Cα coordinates.

In [ ]:
# Step 6 — Python: Build a PDB parser from scratch
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Atom:
    serial:    int
    name:      str   # e.g. 'CA', 'N', 'C', 'O'
    altloc:    str   # alternate location indicator
    res_name:  str   # e.g. 'ALA'
    chain:     str   # chain ID
    res_seq:   int   # residue sequence number
    icode:     str   # insertion code
    x: float
    y: float
    z: float
    occupancy: float
    b_factor:  float
    element:   str
    record:    str   # 'ATOM' or 'HETATM'

    @property
    def coord(self) -> np.ndarray:
        return np.array([self.x, self.y, self.z])

def parse_pdb_line(line: str) -> Optional[Atom]:
    """Parse a single ATOM or HETATM record from a PDB line."""
    record = line[0:6].strip()
    if record not in ('ATOM', 'HETATM'):
        return None
    try:
        serial   = int(line[6:11])
        name     = line[12:16].strip()
        altloc   = line[16].strip()
        res_name = line[17:20].strip()
        chain    = line[21].strip()
        res_seq  = int(line[22:26])
        icode    = line[26].strip()
        x        = float(line[30:38])
        y        = float(line[38:46])
        z        = float(line[46:54])
        occupancy = float(line[54:60]) if line[54:60].strip() else 1.0
        b_factor  = float(line[60:66]) if line[60:66].strip() else 0.0
        element   = line[76:78].strip() if len(line) > 76 else ''
    except (ValueError, IndexError):
        return None
    return Atom(serial, name, altloc, res_name, chain, res_seq, icode,
                x, y, z, occupancy, b_factor, element, record)

def parse_pdb(pdb_text: str) -> List[Atom]:
    """Parse all ATOM/HETATM records from a PDB text string."""
    atoms = []
    for line in pdb_text.splitlines():
        atom = parse_pdb_line(line)
        if atom is not None:
            # Skip alternate conformations (keep only blank or 'A')
            if atom.altloc not in ('', 'A'):
                continue
            atoms.append(atom)
    return atoms

# ---- Simulate a synthetic PDB for a 40-residue helix ----
def generate_pdb_helix(n_residues=40, seq=None):
    """Generate PDB text for an ideal alpha helix."""
    if seq is None:
        seq = ['ALA'] * n_residues
    lines = ['REMARK  Synthetic ideal alpha helix']
    serial = 1
    rise   = 1.5
    radius = 2.3
    res_per_turn = 3.6
    # Simulate backbone atoms N, CA, C, O per residue
    for i, res_name in enumerate(seq):
        angle  = 2 * np.pi * i / res_per_turn
        z      = i * rise
        ca_x   = radius * np.cos(angle)
        ca_y   = radius * np.sin(angle)
        ca_z   = z
        # N is slightly before CA along z
        atoms_rel = [
            ('N',  ca_x - 0.6, ca_y + 0.5, ca_z - 0.5),
            ('CA', ca_x,       ca_y,        ca_z),
            ('C',  ca_x + 0.8, ca_y - 0.3, ca_z + 0.4),
            ('O',  ca_x + 1.2, ca_y - 0.1, ca_z + 0.8),
        ]
        for aname, ax, ay, az in atoms_rel:
            b = np.random.default_rng(i).uniform(5, 25)
            lines.append(
                f'ATOM  {serial:5d}  {aname:<3s} {res_name} A{i+1:4d}    '
                f'{ax:8.3f}{ay:8.3f}{az:8.3f}  1.00{b:6.2f}          '
            )
            serial += 1
    lines.append('END')
    return '\n'.join(lines)

pdb_text = generate_pdb_helix(40)
atoms = parse_pdb(pdb_text)

# Extract Cα atoms only
ca_atoms = [a for a in atoms if a.name == 'CA' and a.record == 'ATOM']
ca_coords = np.array([a.coord for a in ca_atoms])

print(f'Total atoms parsed: {len(atoms)}')
print(f'Cα atoms: {len(ca_atoms)}')
print(f'Unique chains: {set(a.chain for a in atoms)}')
print(f'Residues: {ca_atoms[0].res_seq} – {ca_atoms[-1].res_seq}')
print()
# Radius of gyration
centroid = ca_coords.mean(axis=0)
rg = np.sqrt(((ca_coords - centroid)**2).sum(axis=1).mean())
print(f'Radius of gyration (Cα): {rg:.2f} Å')
# Mean B-factor
mean_b = np.mean([a.b_factor for a in ca_atoms])
print(f'Mean B-factor (Cα): {mean_b:.1f} Å²')
# Cα–Cα bond lengths
ca_dists = np.linalg.norm(np.diff(ca_coords, axis=0), axis=1)
print(f'Mean Cα–Cα distance: {ca_dists.mean():.2f} ± {ca_dists.std():.2f} Å')

In [ ]:
# Step 7 — Visualization: 3D structure + B-factor plot
from mpl_toolkits.mplot3d import Axes3D
fig = plt.figure(figsize=(14, 5))

# Panel A: Cα trace colored by B-factor
ax1 = fig.add_subplot(131, projection='3d')
b_factors = np.array([a.b_factor for a in ca_atoms])
sc = ax1.scatter(ca_coords[:, 0], ca_coords[:, 1], ca_coords[:, 2],
                   c=b_factors, cmap='coolwarm', s=30, alpha=0.9)
ax1.plot(ca_coords[:, 0], ca_coords[:, 1], ca_coords[:, 2],
          'k-', lw=0.8, alpha=0.3)
plt.colorbar(sc, ax=ax1, label='B-factor (Å²)', shrink=0.6)
ax1.set_title('A. Cα trace\n(colored by B-factor)')
ax1.set_xlabel('X (Å)'); ax1.set_ylabel('Y (Å)'); ax1.set_zlabel('Z (Å)')

# Panel B: B-factor per residue
ax2 = fig.add_subplot(132)
res_nums = [a.res_seq for a in ca_atoms]
ax2.plot(res_nums, b_factors, 'o-', ms=3, color='steelblue', lw=1.2)
ax2.axhline(30, color='orange', ls='--', label='B=30 Å² reference')
ax2.set_xlabel('Residue number')
ax2.set_ylabel('B-factor (Å²)')
ax2.set_title('B. B-factor per residue')
ax2.legend(fontsize=8)

# Panel C: Cα–Cα sequential distances (should be ~3.8 Å)
ax3 = fig.add_subplot(133)
ax3.plot(range(1, len(ca_dists) + 1), ca_dists, 'o-', ms=3, color='tomato', lw=1.2)
ax3.axhline(3.8, color='black', ls='--', label='Ideal Cα–Cα (3.8 Å)')
ax3.set_xlabel('Residue pair index')
ax3.set_ylabel('Cα–Cα distance (Å)')
ax3.set_title('C. Sequential Cα–Cα distances\n(should be ~3.8 Å)')
ax3.legend(fontsize=8)

plt.suptitle('Module 11 NB03: PDB Format and Structure Parsing', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pdb_parsing.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement `parse_pdb_line()` from scratch. Test it on the following line:
   ```
   ATOM      1  N   ALA A   1      -0.525   1.362   0.000  1.00 12.31           N
   ```
2. Write a function `get_backbone_atoms(atoms)` that returns a dictionary mapping
   residue number → `{'N': coord, 'CA': coord, 'C': coord, 'O': coord}`.
3. Compute the mean B-factor separately for backbone atoms (N, CA, C, O) vs.
   all atoms. Which is higher and why?
4. What would happen to your parser if a residue has an insertion code
   (e.g. residue "100A")? Fix it.

---
## Step 10 — Quiz

1. What is the difference between ATOM and HETATM records?
2. What does the B-factor represent physically?
3. What columns in an ATOM record give the 3D coordinates?
4. What is the difference between the PDB format and mmCIF format?
5. What does `altloc 'B'` in a PDB record mean?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the PDB file format has limitations that
> led to the development of mmCIF, and name two specific scenarios where
> the legacy PDB format fails.]*

---
*Next: `04_secondary_structure_and_ramachandran.ipynb`*